In [12]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
import os


In [13]:
load_dotenv() 
# print(os.getenv("GEMINI_API_KEY"))

True

In [14]:
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.getenv("GEMINI_API_KEY"),
    temperature=0.7,
)

In [15]:
class LLMState(TypedDict):
    question: str
    answer: str

In [16]:
def llm_qa(state: LLMState) -> LLMState:

    #extract the question from the state
    question = state['question']
    
    #form a prompt for the model
    prompt = f"Answer the question: {question}"
    
    #ask that question to the LLM
    response = model.invoke(prompt).content
    
    #update the answer in the state
    state['answer'] = response
    
    return state

In [17]:
graph = StateGraph(LLMState)

graph.add_node('llm_qa', llm_qa)

graph.add_edge(START, 'llm_qa')

graph.add_edge('llm_qa', END)

workflow = graph.compile()

In [18]:
initial_state = {'question': 'What is the capital of France?'}

final_state = workflow.invoke(initial_state)

print(final_state['answer'])

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (INVALID_ARGUMENT): 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API key not valid. Please pass a valid API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'generativelanguage.googleapis.com'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'API key not valid. Please pass a valid API key.'}]}}